# Importaciones y Configuración Inicial

In [9]:
import pandas as pd
import numpy as np
import psycopg2
from psycopg2 import OperationalError

DB_HOST = "seal-users.c180so2u4aci.us-east-2.rds.amazonaws.com"
DB_NAME = "OLAP_Operations"
DB_USER = "postgres"
DB_PASS = "Nomelase123+"

def obtener_conexion():
    """Establece y devuelve la conexión a la base de datos PostgreSQL."""
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            database=DB_NAME,
            user=DB_USER,
            password=DB_PASS,
            port=5432
        )
        return conn
    except OperationalError as e:
        print(f"Error crítico de conexión: {e}")
        return None

def extraer_datos_interacciones(conn):
    """Extrae las interacciones crudas calculando la bandera lógica de skip desde el OLAP."""
    query = """
        SELECT 
            c.genero,
            c.tema,
            i.id_interaccion,
            CASE 
                WHEN i.tiempo_reproduccion < (c.duracion * 0.3) OR i.dio_dislike = 1 THEN 1.0 
                ELSE 0.0 
            END AS es_skip
        FROM hechos_interacciones i
        JOIN dim_cancion c ON i.id_cancion = c.id_cancion;
    """
    print(" Extrayendo interacciones para cálculo de Skip Rate...")
    try:
        df = pd.read_sql_query(query, conn)
        return df
    except Exception as e:
        print(f" Error al ejecutar la consulta en el modelo: {e}")
        return pd.DataFrame()

def calcular_metricas_segmentadas(df_interacciones):
    """Agrupa por {Tema, Género} y calcula los estadísticos de Z-Score."""
    if df_interacciones.empty:
        return pd.DataFrame()
    
    print(" Agrupando por combinaciones de {Tema, Género}...")
    # Agrupación y cálculo de tasas
    df_segmentos = df_interacciones.groupby(['tema', 'genero']).agg(
        skip_rate=('es_skip', 'mean'),
        total_reproducciones=('id_interaccion', 'count')
    ).reset_index()

    skip_rate_promedio_global = df_segmentos['skip_rate'].mean()
    sigma_global = df_segmentos['skip_rate'].std()
    
    print(f"\n [Métricas Globales] Skip Rate Medio: {skip_rate_promedio_global:.2%} | Desviación Estándar (σ): {sigma_global:.2%}")

    if sigma_global == 0 or np.isnan(sigma_global):
        sigma_global = 0.0001

    df_segmentos['Z_Score'] = (df_segmentos['skip_rate'] - skip_rate_promedio_global) / sigma_global
    df_segmentos['Estatus'] = np.where(
        df_segmentos['Z_Score'] > 2, 'Críticamente mal alineado ',
        np.where(df_segmentos['Z_Score'] < -2, 'Alta Retención ', 'Normal')
    )
    
    return df_segmentos


conexion = obtener_conexion()

if conexion:
    df_raw = extraer_datos_interacciones(conexion)
    conexion.close() 
    
    df_analisis = calcular_metricas_segmentadas(df_raw)
    print("\n Procesamiento completado con éxito.")
    

    if not df_analisis.empty:

        puntos_friccion = df_analisis[df_analisis['Z_Score'] > 2].sort_values(by='Z_Score', ascending=False)

        print("\n" + "="*80)
        print("   REPORTE DE PUNTOS DE FRICCIÓN DE CONTENIDO (ANOMALÍAS DE CHURN)")
        print("="*80)
        print(f"Se evaluaron {len(df_analisis)} pares únicos de {{Tema, Género}}.")
        print(f"Se encontraron {len(puntos_friccion)} combinaciones críticamente desalineadas.\n")

        if not puntos_friccion.empty:

            print(puntos_friccion[['tema', 'genero', 'skip_rate', 'Z_Score', 'Estatus']].to_string(index=False))
        else:
            print(" ¡Excelente estabilidad! Ningún segmento supera el umbral de anomalía crítica (|Z| > 2).")
        print("="*80)


        archivo_salida = "reporte_anomalias_olap.csv"
        df_analisis.to_csv(archivo_salida, index=False)
        print(f" Base de datos completa procesada y guardada en: '{archivo_salida}'\n")
        
else:
    print(" No se pudo iniciar el pipeline debido a fallas de conexión.")

 Extrayendo interacciones para cálculo de Skip Rate...


C:\Users\luigu\AppData\Local\Temp\ipykernel_20816\1464563904.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


 Agrupando por combinaciones de {Tema, Género}...

 [Métricas Globales] Skip Rate Medio: 58.21% | Desviación Estándar (σ): 0.63%

 Procesamiento completado con éxito.

   REPORTE DE PUNTOS DE FRICCIÓN DE CONTENIDO (ANOMALÍAS DE CHURN)
Se evaluaron 140 pares únicos de {Tema, Género}.
Se encontraron 5 combinaciones críticamente desalineadas.

      tema     genero  skip_rate  Z_Score                    Estatus
      Amor       Jazz   0.603020 3.333212 Críticamente mal alineado 
        Fe       Rock   0.599078 2.705648 Críticamente mal alineado 
      Amor Electronic   0.597054 2.383355 Críticamente mal alineado 
Naturaleza        R&B   0.595000 2.056306 Críticamente mal alineado 
  Protesta       Folk   0.594929 2.044998 Críticamente mal alineado 
 Base de datos completa procesada y guardada en: 'reporte_anomalias_olap.csv'

